## **Stochastic Gradient Descent (SGD)**

### **Topic Roadmap**

 **1. Imports & Dataset Preparation**

 **2. OLS Baseline Model**

 **3. Custom Stochastic Gradient Descent**

 **4. Scikit-Learn SGD Implementation**

 **5. Key Revision Notes**

### **1. Imports & Dataset Preparation**

Import the required libraries, load the diabetes regression dataset, and split it into training and testing sets[cite: 4].

In [1]:
import time
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Load diabetes dataset
X, y = load_diabetes(return_X_y=True)

# Create training and testing splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

### **2. OLS Baseline Model**

Train a standard Ordinary Least Squares (OLS) Linear Regression model to establish a baseline $R^2$ score[cite: 4].

In [2]:
# Initialize and train the baseline model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Evaluate baseline performance
y_pred_ols = lr_model.predict(X_test)
baseline_r2 = r2_score(y_test, y_pred_ols)

print(f"Baseline OLS R2 Score: {baseline_r2:.4f}")

Baseline OLS R2 Score: 0.4399


### **3. Custom Stochastic Gradient Descent**

Build a custom Stochastic Gradient Descent regressor from scratch. 

Unlike Batch Gradient Descent, SGD updates the coefficients and intercept by calculating the loss gradient for a **single, randomly selected sample** at each step of the epoch[cite: 4].

In [3]:
class CustomSGDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        
    def fit(self, X_train, y_train):
        # Initialize intercept to 0 and coefficients to an array of ones
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])

        # Iterate over the specified number of epochs
        for i in range(self.epochs):
            # Perform an update for every sample in the dataset
            for j in range(X_train.shape[0]):
                # Randomly select a single sample index
                idx = np.random.randint(0, X_train.shape[0])

                # Calculate prediction for the single sample
                y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_
                
                # Calculate loss gradients with respect to intercept and coefficients
                intercept_der = -2 * (y_train[idx] - y_hat)
                coef_der = -2 * np.dot((y_train[idx] - y_hat), X_train[idx])

                # Apply parameter updates instantly
                self.intercept_ -= (self.lr * intercept_der)
                self.coef_ -= (self.lr * coef_der)
                
    def predict(self, X_test):
        # Calculate final predictions using the learned parameters
        return np.dot(X_test, self.coef_) + self.intercept_

Train the custom SGD model, evaluate its performance, and measure the training time[cite: 4].

In [4]:
# Initialize the custom model
custom_sgd = CustomSGDRegressor(learning_rate=0.01, epochs=50)

# Train and measure execution time
start_time = time.time()
custom_sgd.fit(X_train, y_train)
training_time = time.time() - start_time

# Evaluate custom model performance
y_pred_custom = custom_sgd.predict(X_test)
custom_r2 = r2_score(y_test, y_pred_custom)

print(f"Custom SGD R2 Score: {custom_r2:.4f}")
print(f"Training Time: {training_time:.4f} seconds")

Custom SGD R2 Score: 0.4309
Training Time: 0.7922 seconds


### **4. Scikit-Learn SGD Implementation**

Compare the custom implementation against `sklearn.linear_model.SGDRegressor` utilizing a constant learning rate[cite: 4].

In [5]:
# Initialize and train the scikit-learn SGD model
sklearn_sgd = SGDRegressor(max_iter=100, learning_rate='constant', eta0=0.01)
sklearn_sgd.fit(X_train, y_train)

# Evaluate sklearn model performance
y_pred_sklearn = sklearn_sgd.predict(X_test)
sklearn_r2 = r2_score(y_test, y_pred_sklearn)

print(f"Sklearn SGD R2 Score: {sklearn_r2:.4f}")

Sklearn SGD R2 Score: 0.4305


### **5. Key Revision Notes**

- **Stochastic Gradient Descent (SGD):** An optimization method where weights are updated iteratively using the gradient of the loss function computed from a *single, randomly chosen training sample* rather than the whole dataset[cite: 4].
- **Random Selection:** The use of `np.random.randint()` ensures that the algorithm navigates the loss landscape stochastically[cite: 4], which helps it escape local minima in non-convex functions.
- **Execution Trade-offs:** SGD performs many rapid parameter updates making it memory-efficient and scalable for massive datasets, though its path to convergence is much noisier compared to Batch Gradient Descent.